# Đề tài 4 — P2 (`svmlib`): Chạy kiểm thử đối chiếu LIBSVM & Thí nghiệm C5 trên Google Colab

Notebook này giúp bạn chạy toàn bộ bộ test **A4 (so với LIBSVM / `scikit-learn`)**, xuất bảng so sánh cho báo cáo, tạo các đồ thị vector PDF (`A1_convergence.pdf`, `A3_calibration.pdf`, `C5_timing.pdf`), và thử tích hợp `svmlib.SVC` vào `sklearn.pipeline.Pipeline` cho P1 và P3.

**Cách dùng nhanh:**
1. Upload file `P2_svmlib_deliverables.zip` lên Colab (biểu tượng thư mục bên trái) **hoặc** chạy ô clone repo GitHub của nhóm.
2. Chọn **Runtime → Run all** để chạy toàn bộ.

In [ ]:
# 1. Giải nén gói P2 (hoặc bỏ comment dòng git clone nếu bạn đã push lên GitHub)
import os
if os.path.exists('P2_svmlib_deliverables.zip'):
    !unzip -oq P2_svmlib_deliverables.zip -d p2_workspace
    %cd p2_workspace
# !git clone https://github.com/Solenfw/ML-SVM_and_KernelMethods.git
# %cd ML-SVM_and_KernelMethods

!pip -q install pytest numba
!ls -la

## Bước 1 (A4): Chạy toàn bộ unit test `pytest` đối chiếu với LIBSVM, LinearSVC và KernelRidge

In [ ]:
!PYTHONPATH=. pytest -v tests/test_vs_libsvm.py

## Bước 2 (A4): Xuất bảng số liệu đối chiếu LIBSVM (`experiments/a4_libsvm_comparison.csv`)

In [ ]:
!PYTHONPATH=. python experiments/a4_report_table.py

## Bước 3 (A1 & A3): Tạo đồ thị hội tụ SMO (`A1_convergence.pdf`) và Reliability Diagram (`A3_calibration.pdf`)

In [ ]:
!PYTHONPATH=. python experiments/generate_a1_a3_figures.py

## Bước 4 (C5): Đo thời gian SMO tuyến tính vs DCD ($n = 10^3 \dots 10^5$, tự động dùng Numba trên Colab)

Mặc định chạy `--repeats 3 --timeout 1800` (giới hạn 30 phút/lần theo đúng kế hoạch). Bạn có thể chỉnh `--timeout 120` để chạy thử nhanh trước.

In [ ]:
!PYTHONPATH=. python experiments/c5_timing.py --repeats 3 --tol 1e-2 --timeout 300
import pandas as pd
pd.read_csv('experiments/c5_timing_results.csv')

## Bước 5: Ví dụ tích hợp `svmlib.SVC` vào `sklearn` Pipeline + GridSearchCV cho P1 và P3

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from svmlib import SVC

X, y = load_iris(return_X_y=True)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(C=1.0, kernel='rbf', gamma='scale', decision_function_shape='ovo', probability=True)),
])
scores = cross_val_score(pipe, X, y, cv=5)
print(f'5-fold CV accuracy trên Iris: {scores.mean():.4f} ± {scores.std():.4f}')
pipe.fit(X, y)
print('5 mẫu xác suất Platt + Pairwise Coupling đầu tiên:\n', pipe.predict_proba(X[:5]).round(3))